In [2]:
from typing import List
from datetime import date
from sqlalchemy import String, Integer, ForeignKey, DECIMAL, Date
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship
from sqlalchemy import create_engine, inspect, select
from sqlalchemy.orm import Session
from sqlalchemy import inspect
from sqlalchemy import text
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from adbc_driver_postgresql import dbapi
import pandas as pd
import os
import plotly.express as px
import plotly.io as pio
import pandas as pd

pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)
pd.options.plotting.backend = 'plotly'
pio.templates.default = 'plotly_dark+presentation'

# --- Configuration ---
BASE_DIR = Path('./')
ENV_PATH = Path().cwd().parent/"creds.env"
load_dotenv(dotenv_path=ENV_PATH)

DB_USER = os.getenv("DB_USER", "postgres")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_NAME = os.getenv("DB_NAME", "unarxive")

CONNECTION_STRING = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}"
engine = create_engine(CONNECTION_STRING)

In [10]:
print(DB_HOST)
print(DB_USER)
print(DB_PASS)
print(Path().cwd())

147.126.2.24
mgarciamelo
cotsEk-5padby-jivgib
/home/jupyter-mgarciamelo/ipynb


In [2]:
def run_query(query: str, engine=engine) -> pd.DataFrame:
  return pd.read_sql(query, engine)

In [3]:
sql = """
WITH top_cited AS (
    SELECT 
        citation_patent_id,
        COUNT(DISTINCT(patent_id)) AS citation_count
    FROM g_us_patent_citation
    GROUP BY citation_patent_id
    ORDER BY citation_count DESC
    LIMIT 20
)
SELECT *
FROM top_cited t
JOIN g_patent p ON t.citation_patent_id = p.patent_id
JOIN g_attorney_disambiguated a ON t.citation_patent_id = a.patent_id
JOIN g_assignee_disambiguated asg ON t.citation_patent_id = asg.patent_id
ORDER BY t.citation_count DESC
"""

display(pd.read_sql(text(sql), engine))

,citation_patent_id,citation_count,patent_id,patent_type,patent_date,patent_title,wipo_kind,num_claims,withdrawn,filename,patent_id,attorney_sequence,attorney_id,disambig_attorney_name_first,disambig_attorney_name_last,disambig_attorney_organization,attorney_country,patent_id,assignee_sequence,assignee_id,disambig_assignee_individual_name_first,disambig_assignee_individual_name_last,disambig_assignee_organization,assignee_type,location_id
0,4683202,5784,4683202,utility,1987-07-28,Process for amplifying nucleic acid sequences,A,21,0,pftaps19870728_wk30.zip,4683202,1,574539b9c6e6fefa36bfb4a84f0588f4,Albert P.,Halluin,,,4683202,0,be1a71b8-c282-44cf-ae7b-a82d30a93619,None,None,Cetus Corporation,2.0,c7f663aa-16c7-11ed-9b5f-1234bde3cd05
1,4683202,5784,4683202,utility,1987-07-28,Process for amplifying nucleic acid sequences,A,21,0,pftaps19870728_wk30.zip,4683202,0,55f611002a893a054ebd81aafbf10072,Janet E.,Hasak,,,4683202,0,be1a71b8-c282-44cf-ae7b-a82d30a93619,None,None,Cetus Corporation,2.0,c7f663aa-16c7-11ed-9b5f-1234bde3cd05
2,4683195,5238,4683195,utility,1987-07-28,"Process for amplifying, detecting, and/or-clon...",A,26,0,pftaps19870728_wk30.zip,4683195,0,55f611002a893a054ebd81aafbf10072,Janet E.,Hasak,,,4683195,0,be1a71b8-c282-44cf-ae7b-a82d30a93619,None,None,Cetus Corporation,2.0,c7f663aa-16c7-11ed-9b5f-1234bde3cd05
3,4683195,5238,4683195,utility,1987-07-28,"Process for amplifying, detecting, and/or-clon...",A,26,0,pftaps19870728_wk30.zip,4683195,1,574539b9c6e6fefa36bfb4a84f0588f4,Albert P.,Halluin,,,4683195,0,be1a71b8-c282-44cf-ae7b-a82d30a93619,None,None,Cetus Corporation,2.0,c7f663aa-16c7-11ed-9b5f-1234bde3cd05
4,5523520,4891,5523520,utility,1996-06-04,Mutant dwarfism gene of petunia,A,20,0,pftaps19960604_wk23.zip,5523520,0,849a92c42d1623421524dbc26fc44ea0,,,"Rothwell, Figg, Ernst & Kurz",,5523520,0,3be2dc91-4af7-4603-b9c9-e36b04464e8e,None,None,"Goldsmith Seeds, Inc.",2.0,15a22f0f-16c8-11ed-9b5f-1234bde3cd05
5,7674650,4307,7674650,utility,2010-03-09,Semiconductor device and manufacturing method ...,B2,28,0,ipg100309.xml,7674650,0,d0ec2435db8a518d82eeb5cd905c6f39,,,Cook Alex Ltd.,,7674650,0,3cbd0e04-001d-46a5-986d-456478d9e140,None,None,"Semiconductor Energy Laboratory Co., Ltd.",3.0,7159b2a8-16c8-11ed-9b5f-1234bde3cd05
6,7061014,4050,7061014,utility,2006-06-13,Natural-superlattice homologous single crystal...,B2,12,0,ipg060613.xml,7061014,0,b2a3fc49e9820b0980302263ef914c24,,,"Weterman, Hattori, Daniels & Adrian, LLP.",,7061014,0,eb2ebb85-8503-414d-9730-aed37c7ab928,None,None,Japan Science and Technology Agency,3.0,a430f863-16c8-11ed-9b5f-1234bde3cd05
7,5731856,4035,5731856,utility,1998-03-24,Methods for forming liquid crystal displays in...,A,25,0,pftaps19980324_wk12.zip,5731856,0,42891b44c6faa644f70ef9c230ff1be3,,,"Myers, Bigel, Sibley & Sajovec",,5731856,0,d1d96a91-c5e6-4090-ba39-4af2494eb35c,None,None,"SAMSUNG ELECTRONICS CO., LTD.",3.0,3eea8bf8-16c8-11ed-9b5f-1234bde3cd05
8,7732819,4030,7732819,utility,2010-06-08,Semiconductor device and manufacturing method ...,B2,49,0,ipg100608.xml,7732819,0,c6bde8e25976dd208350aad111ed92e8,,,Husch Blackwell Sanders,,7732819,0,3cbd0e04-001d-46a5-986d-456478d9e140,None,None,"Semiconductor Energy Laboratory Co., Ltd.",3.0,7159b2a8-16c8-11ed-9b5f-1234bde3cd05
9,6727522,3992,6727522,utility,2004-04-27,Transistor and semiconductor device,B1,44,0,pg040427.zip,6727522,0,60001a7c83725e6a3b883b37fbc5bcb5,,,"Neifeld IP Law, P.C.",,6727522,0,7f546aca-e348-461a-8460-ca6146537efb,None,None,Japan Science and Technology,3.0,34b4468e-16c8-11ed-9b5f-1234bde3cd05


In [ ]:
sql = """
WITH top_cited AS (
    SELECT 
        citation_patent_id,
        COUNT(DISTINCT(patent_id)) AS citation_count
    FROM g_us_patent_citation
    GROUP BY citation_patent_id
    ORDER BY citation_count DESC
    LIMIT 20
)
SELECT *
FROM top_cited t
JOIN g_patent p ON t.citation_patent_id = p.patent_id
JOIN g_cpc_current c ON t.citation_patent_id = c.patent_id
ORDER BY t.citation_count DESC
"""
display(pd.read_sql(text(sql), engine))

In [5]:
sql = """
WITH top_cited AS (
    SELECT citation_patent_id,
           COUNT(DISTINCT patent_id) AS citation_count
    FROM g_us_patent_citation c
    JOIN g_patent p ON c.citation_patent_id = p.patent_id
    WHERE EXTRACT(YEAR FROM p.patent_date) > 2004
    GROUP BY citation_patent_id
    ORDER BY citation_count DESC
    LIMIT 5
)
SELECT *
FROM top_cited t
JOIN g_patent p ON t.citation_patent_id = p.patent_id
JOIN g_cpc_current c ON t.citation_patent_id = c.patent_id
WHERE c.cpc_sequence = 0
ORDER BY t.citation_count DESC;
"""

df = pd.read_sql(text(sql), engine)
display(df)

ProgrammingError: (psycopg2.errors.AmbiguousColumn) column reference "patent_id" is ambiguous
LINE 4:            COUNT(DISTINCT patent_id) AS citation_count
                                  ^

[SQL: 
WITH top_cited AS (
    SELECT citation_patent_id,
           COUNT(DISTINCT patent_id) AS citation_count
    FROM g_us_patent_citation c
    JOIN g_patent p ON c.citation_patent_id = p.patent_id
    WHERE EXTRACT(YEAR FROM p.patent_date) > 2004
    GROUP BY citation_patent_id
    ORDER BY citation_count DESC
    LIMIT 5
)
SELECT *
FROM top_cited t
JOIN g_patent p ON t.citation_patent_id = p.patent_id
JOIN g_cpc_current c ON t.citation_patent_id = c.patent_id
WHERE c.cpc_sequence = 0
ORDER BY t.citation_count DESC;
]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [7]:
sql = '''
SELECT loc.disambig_state, 
       COUNT(DISTINCT CONCAT(c.patent_id, '-', c.citation_patent_id)) AS citation_count
FROM g_us_patent_citation c
JOIN g_patent p ON c.citation_patent_id = p.patent_id
JOIN g_inventor_disambiguated inv ON p.patent_id = inv.patent_id
JOIN g_location_disambiguated loc ON inv.location_id = loc.location_id
WHERE EXTRACT(YEAR FROM p.patent_date) > 2004
  AND loc.disambig_country = 'US'
  AND loc.disambig_state IS NOT NULL
GROUP BY loc.disambig_state
ORDER BY citation_count DESC
LIMIT 50;
'''
df = pd.read_sql(text(sql), engine)
display(df)

,disambig_state,citation_count
0,CA,11736523
1,OH,3012989
2,MA,2417866
3,NY,2330397
4,TX,2102936
5,WA,1945780
6,CT,1839070
7,NJ,1531188
8,MI,1428757
9,PA,1376923


In [3]:
sql = """
SELECT EXTRACT(YEAR FROM p.patent_date) AS year,
    COUNT(DISTINCT CONCAT(c.patent_id, '-', c.citation_patent_id)) AS citation_count
FROM g_us_patent_citation c
JOIN g_patent p ON c.citation_patent_id = p.patent_id
WHERE EXTRACT(YEAR FROM p.patent_date) > 2004
GROUP BY year
ORDER BY citation_count DESC;
"""
pd.read_sql(text(sql), engine)

,year,citation_count
0,2006.0,4186766
1,2010.0,3832887
2,2012.0,3553211
3,2013.0,3551660
4,2005.0,3521116
5,2007.0,3507857
6,2011.0,3482871
7,2014.0,3368211
8,2008.0,3206258
9,2009.0,3104452


In [4]:
sql = """
SELECT cpc_class, COUNT(DISTINCT CONCAT(c.patent_id, '-', c.citation_patent_id)) AS citation_count
FROM g_us_patent_citation c
JOIN g_patent p ON c.citation_patent_id = p.patent_id
JOIN g_cpc_current cpc ON p.patent_id = cpc.patent_id
WHERE EXTRACT(YEAR FROM p.patent_date) > 2004
GROUP BY cpc_class
ORDER BY citation_count DESC
LIMIT 20;
"""
pd.read_sql(text(sql), engine)

,cpc_class,citation_count
0,A61,12208715
1,G06,9180074
2,H04,7666218
3,Y10,4438450
4,H01,3629082
5,G01,2881646
6,Y02,2724062
7,H10,1732216
8,G02,1341204
9,B60,1318625


In [12]:
sql = """
SELECT *
FROM g_cpc_current cpc
WHERE cpc.patent_id = '10000271';
"""

pd.read_sql(text(sql), engine)


,patent_id,cpc_sequence,cpc_section,cpc_class,cpc_subclass,cpc_group,cpc_type
0,10000271,0,B,B64,B64C,B64C1/1461,inventional
1,10000271,1,B,B64,B64C,B64C1/1423,inventional
2,10000271,2,E,E06,E06B,E06B7/23,inventional
3,10000271,3,B,B60,B60J,B60J10/244,additional


In [4]:
sql = """
SELECT *
FROM g_location_disambiguated l
JOIN g_inventor_disambiguated i ON l.location_id = i.location_id
JOIN g_patent p ON i.patent_id = p.patent_id
WHERE p.patent_id = '7674650';
"""

pd.read_sql(text(sql), engine)

,location_id,disambig_city,disambig_state,disambig_country,latitude,longitude,county,state_fips,county_fips,patent_id,inventor_sequence,inventor_id,disambig_inventor_name_first,disambig_inventor_name_last,gender_code,location_id,patent_id,patent_type,patent_date,patent_title,wipo_kind,num_claims,withdrawn,filename
0,1d2251c8-16c8-11ed-9b5f-1234bde3cd05,Yokohama,None,JP,35.444991,139.636768,None,None,None,7674650,0,fl:ke_ln:akimoto-8,Kengo,Akimoto,M,1d2251c8-16c8-11ed-9b5f-1234bde3cd05,7674650,utility,2010-03-09,Semiconductor device and manufacturing method ...,B2,28,0,ipg100309.xml
1,1d2251c8-16c8-11ed-9b5f-1234bde3cd05,Yokohama,None,JP,35.444991,139.636768,None,None,None,7674650,2,fl:no_ln:sone-5,Norihito,Sone,M,1d2251c8-16c8-11ed-9b5f-1234bde3cd05,7674650,utility,2010-03-09,Semiconductor device and manufacturing method ...,B2,28,0,ipg100309.xml
2,1d2251c8-16c8-11ed-9b5f-1234bde3cd05,Yokohama,None,JP,35.444991,139.636768,None,None,None,7674650,1,fl:ta_ln:honda-37,Tatsuya,Honda,M,1d2251c8-16c8-11ed-9b5f-1234bde3cd05,7674650,utility,2010-03-09,Semiconductor device and manufacturing method ...,B2,28,0,ipg100309.xml


In [10]:
sql = """
SELECT cpc_class, cpc_class_title, COUNT(*) as subclasses
FROM g_cpc_title
GROUP BY cpc_class, cpc_class_title
ORDER BY subclasses DESC
LIMIT 10;
"""

pd.read_sql(text(sql), engine)

,cpc_class,cpc_class_title,subclasses
0,H01,ELECTRIC ELEMENTS,20504
1,Y10,TECHNICAL SUBJECTS COVERED BY FORMER USPC,16049
2,A61,MEDICAL OR VETERINARY SCIENCE; HYGIENE,13709
3,C12,BIOCHEMISTRY; BEER; SPIRITS; WINE; VINEGAR; MI...,10479
4,H04,ELECTRIC COMMUNICATION TECHNIQUE,10321
5,G01,MEASURING; TESTING,10053
6,G05,CONTROLLING; REGULATING,9318
7,B65,CONVEYING; PACKING; STORING; HANDLING THIN OR ...,9019
8,B01,PHYSICAL OR CHEMICAL PROCESSES OR APPARATUS IN...,8475
9,B60,VEHICLES IN GENERAL,8389


In [9]:
sql = '''
SELECT COUNT(DISTINCT patent_id)
FROM g_patent p
WHERE EXTRACT(YEAR FROM p.patent_date) > 2004
'''

pd.read_sql(text(sql), engine)

,count
0,6188004


In [7]:
sql = '''
SELECT COUNT(c.patent_id)
FROM g_us_patent_citation c
JOIN g_patent p ON c.patent_id = p.patent_id
WHERE EXTRACT(YEAR FROM p.patent_date) > 2004
'''

pd.read_sql(text(sql), engine)

,count
0,122082531


In [8]:
sql = '''
SELECT AVG(cite_count)
FROM (
    SELECT c.citation_patent_id, COUNT(*) AS cite_count
    FROM g_us_patent_citation c
    JOIN g_patent p ON c.citation_patent_id = p.patent_id
    WHERE EXTRACT(YEAR FROM p.patent_date) > 2004
    GROUP BY c.citation_patent_id
) sub

'''
pd.read_sql(text(sql), engine)

,avg
0,13.827994


In [9]:
sql  = '''
SELECT *
FROM g_us_patent_citation c
JOIN g_patent p ON c.patent_id = p.patent_id
JOIN g_assignee_disambiguated a ON p.patent_id = a.patent_id
WHERE disambig_assignee_organization LIKE '%Apple%'
AND EXTRACT(YEAR FROM p.patent_date) = 2006
LIMIT 100;
'''
pd.read_sql(text(sql), engine)


,patent_id,citation_sequence,citation_patent_id,citation_date,record_name,citation_category,patent_id,patent_type,patent_date,patent_title,wipo_kind,num_claims,withdrawn,filename,patent_id,assignee_sequence,assignee_id,disambig_assignee_individual_name_first,disambig_assignee_individual_name_last,disambig_assignee_organization,assignee_type,location_id
0,7083974,0,5333675,1994-08-01,Mullis et al.,cited by other,7083974,utility,2006-08-01,Rotatable sample disk and method of loading a ...,B2,19,0,ipg060801.xml,7083974,0,03d6786b-977a-483d-b3cd-3bf377f4174c,None,None,Applera Corporation,2.0,1416e13d-16c8-11ed-9b5f-1234bde3cd05
1,7083974,1,5475610,1995-12-01,Atwood et al.,cited by other,7083974,utility,2006-08-01,Rotatable sample disk and method of loading a ...,B2,19,0,ipg060801.xml,7083974,0,03d6786b-977a-483d-b3cd-3bf377f4174c,None,None,Applera Corporation,2.0,1416e13d-16c8-11ed-9b5f-1234bde3cd05
2,7083974,2,5656493,1997-08-01,Mullis et al.,cited by other,7083974,utility,2006-08-01,Rotatable sample disk and method of loading a ...,B2,19,0,ipg060801.xml,7083974,0,03d6786b-977a-483d-b3cd-3bf377f4174c,None,None,Applera Corporation,2.0,1416e13d-16c8-11ed-9b5f-1234bde3cd05
3,7083974,3,5693233,1997-12-01,Schembri,cited by other,7083974,utility,2006-08-01,Rotatable sample disk and method of loading a ...,B2,19,0,ipg060801.xml,7083974,0,03d6786b-977a-483d-b3cd-3bf377f4174c,None,None,Applera Corporation,2.0,1416e13d-16c8-11ed-9b5f-1234bde3cd05
4,7083974,4,5928907,1999-07-01,Woudenberg et al.,cited by other,7083974,utility,2006-08-01,Rotatable sample disk and method of loading a ...,B2,19,0,ipg060801.xml,7083974,0,03d6786b-977a-483d-b3cd-3bf377f4174c,None,None,Applera Corporation,2.0,1416e13d-16c8-11ed-9b5f-1234bde3cd05
5,7083974,5,6015674,2000-01-01,Woudenberg et al.,cited by other,7083974,utility,2006-08-01,Rotatable sample disk and method of loading a ...,B2,19,0,ipg060801.xml,7083974,0,03d6786b-977a-483d-b3cd-3bf377f4174c,None,None,Applera Corporation,2.0,1416e13d-16c8-11ed-9b5f-1234bde3cd05
6,7083974,6,6174670,2001-01-01,Wittwer et al.,cited by other,7083974,utility,2006-08-01,Rotatable sample disk and method of loading a ...,B2,19,0,ipg060801.xml,7083974,0,03d6786b-977a-483d-b3cd-3bf377f4174c,None,None,Applera Corporation,2.0,1416e13d-16c8-11ed-9b5f-1234bde3cd05
7,7083974,7,6303305,2001-10-01,Wittwer et al.,cited by other,7083974,utility,2006-08-01,Rotatable sample disk and method of loading a ...,B2,19,0,ipg060801.xml,7083974,0,03d6786b-977a-483d-b3cd-3bf377f4174c,None,None,Applera Corporation,2.0,1416e13d-16c8-11ed-9b5f-1234bde3cd05
8,7083974,8,6387621,2002-05-01,Wittwer,cited by other,7083974,utility,2006-08-01,Rotatable sample disk and method of loading a ...,B2,19,0,ipg060801.xml,7083974,0,03d6786b-977a-483d-b3cd-3bf377f4174c,None,None,Applera Corporation,2.0,1416e13d-16c8-11ed-9b5f-1234bde3cd05
9,D519731,0,D171247,1954-01-01,Charnota,cited by other,D519731,design,2006-05-02,Electronic device holder,S1,1,0,ipg060502.xml,D519731,0,4c8f5d40-1e37-48b9-8de1-56482301c08b,None,None,Apple Computer Inc.,2.0,c6a61746-16c7-11ed-9b5f-1234bde3cd05
